# Model Implementations (No Black Boxes)

This notebook rebuilds core retrieval scoring ideas from scratch on toy data:
- TF-IDF
- Cosine similarity ranking
- BM25 scoring formula

The goal is to understand model mechanics, not to optimize runtime.


## 1. Toy Corpus and Query


In [ ]:
import math
import re
from collections import Counter, defaultdict

token_pattern = re.compile(r'[a-z0-9]+')


def tokenize(text):
    return token_pattern.findall(text.lower())


docs = [
    'python dataframe merge columns',
    'bm25 ranking for information retrieval',
    'how to merge two dataframes in pandas',
    'cosine similarity with tf idf vectors',
]
query = 'merge dataframe pandas'

doc_tokens = [tokenize(d) for d in docs]
query_tokens = tokenize(query)

print('query tokens:', query_tokens)
print('doc tokens:', doc_tokens)


## 2. TF, DF, and IDF From Scratch


In [ ]:
N = len(doc_tokens)

# Document frequency: in how many documents each term appears.
df = defaultdict(int)
for tokens in doc_tokens:
    for term in set(tokens):
        df[term] += 1

# Smoothed IDF.
idf = {term: math.log((1 + N) / (1 + df_val)) + 1 for term, df_val in df.items()}

print('N documents =', N)
print('sample idf values:')
for term in sorted(list(idf.keys()))[:8]:
    print(f'  {term:12s} -> {idf[term]:.4f}')


## 3. Build TF-IDF Vectors Manually


In [ ]:
vocab = sorted(idf.keys())
term_to_idx = {term: i for i, term in enumerate(vocab)}


def tfidf_vector(tokens):
    counts = Counter(tokens)
    vec = [0.0] * len(vocab)
    for term, count in counts.items():
        if term in term_to_idx:
            i = term_to_idx[term]
            tf = count / len(tokens)
            vec[i] = tf * idf[term]
    return vec


doc_vecs = [tfidf_vector(tokens) for tokens in doc_tokens]
query_vec = tfidf_vector(query_tokens)

print('vocab size:', len(vocab))
print('query vector length:', len(query_vec))


## 4. Cosine Similarity and Ranking


In [ ]:
def dot(a, b):
    return sum(x * y for x, y in zip(a, b))


def norm(a):
    return math.sqrt(dot(a, a))


def cosine(a, b):
    na, nb = norm(a), norm(b)
    if na == 0.0 or nb == 0.0:
        return 0.0
    return dot(a, b) / (na * nb)


scores = [cosine(query_vec, dvec) for dvec in doc_vecs]
ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

for rank, (doc_idx, score) in enumerate(ranked, start=1):
    print(f'{rank}. doc[{doc_idx}] score={score:.4f} | {docs[doc_idx]}')


## 5. BM25 Formula From Scratch

For term `t` in query `q` and document `d`:

`score(d, q) += idf(t) * ((tf * (k1 + 1)) / (tf + k1 * (1 - b + b * |d| / avgdl)))`

where:
- `tf`: term frequency in document
- `|d|`: document length
- `avgdl`: average document length
- `k1`, `b`: hyperparameters


In [ ]:
def bm25_idf(term, docs_tokens):
    n_docs = len(docs_tokens)
    n_with_term = sum(1 for tokens in docs_tokens if term in tokens)
    # BM25 idf variant
    return math.log(1 + (n_docs - n_with_term + 0.5) / (n_with_term + 0.5))


def bm25_score(query_tokens, doc_tokens, docs_tokens, k1=1.5, b=0.75):
    avgdl = sum(len(toks) for toks in docs_tokens) / len(docs_tokens)
    dl = len(doc_tokens)
    tf_counts = Counter(doc_tokens)

    score = 0.0
    for term in query_tokens:
        if term not in tf_counts:
            continue

        tf = tf_counts[term]
        idf_term = bm25_idf(term, docs_tokens)
        numerator = tf * (k1 + 1)
        denominator = tf + k1 * (1 - b + b * (dl / avgdl))
        score += idf_term * (numerator / denominator)

    return score


bm25_scores = [bm25_score(query_tokens, d_tokens, doc_tokens) for d_tokens in doc_tokens]
bm25_ranked = sorted(enumerate(bm25_scores), key=lambda x: x[1], reverse=True)

for rank, (doc_idx, score) in enumerate(bm25_ranked, start=1):
    print(f'{rank}. doc[{doc_idx}] score={score:.4f} | {docs[doc_idx]}')


## 6. What to Take Away

- TF-IDF and BM25 are both term-matching methods but with different term-frequency behavior.
- BM25 controls term-frequency saturation and normalizes for document length.
- This is why BM25 often gives more stable rankings when documents have very different sizes.

In the main project code, these same ideas are applied to the full corpus at scale.


## 7. Embedding-based Retrieval Demo

This demonstrates the third Phase-1 family: semantic retrieval with sentence embeddings.
We encode query and documents into vectors, then rank by cosine similarity.


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

doc_texts = docs
query_text = query

doc_emb = model.encode(doc_texts, normalize_embeddings=True)
query_emb = model.encode([query_text], normalize_embeddings=True)[0]

# cosine similarity for normalized vectors = dot product
dense_scores = doc_emb @ query_emb
dense_ranked = sorted(enumerate(dense_scores), key=lambda x: x[1], reverse=True)

for rank, (doc_idx, score) in enumerate(dense_ranked, start=1):
    print(f'{rank}. doc[{doc_idx}] score={score:.4f} | {doc_texts[doc_idx]}')
